In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import requests
from PIL import Image
from io import BytesIO

# Cargar el archivo JSON guardado por main.py / funciones.py
try:
    with open('datos.json', 'r', encoding='utf-8') as f:
        datos = json.load(f)
    df = pd.DataFrame(datos)
    
    # Calcular columna de valor total acumulado por tipo de carta
    if not df.empty:
        df['valor_total_usd'] = df['precio_usd'] * df['cantidad']
        print(f"✅ Se cargaron {len(df)} registros correctamente.")
        display(df.head())
    else:
        print("⚠️ La colección está vacía. Agregá cartas desde main.py primero.")
except FileNotFoundError:
    print("❌ No se encontró el archivo datos.json. Ejecutá main.py para generarlo.")

In [ ]:
if 'df' in locals() and not df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 1. Gráfico de Barras: Cantidad total de cartas por tipo
    df_tipo = df.groupby('tipo')['cantidad'].sum()
    df_tipo.plot(kind='bar', ax=axes[0], color='#ffcb05', edgecolor='#3657a0', linewidth=1.5)
    axes[0].set_title('Cantidad Total de Cartas por Tipo', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Tipo de Pokémon')
    axes[0].set_ylabel('Copias en Colección')
    axes[0].grid(axis='y', linestyle='--', alpha=0.7)

    # 2. Gráfico de Barras: Valor económico por Rareza
    df_rareza = df.groupby('raraza')['valor_total_usd'].sum()
    df_rareza.plot(kind='barh', ax=axes[1], color='#e3350d', edgecolor='black')
    axes[1].set_title('Valor Total Estimado ($ USD) por Rareza', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Valor Total ($ USD)')
    axes[1].set_ylabel('Rareza')
    axes[1].grid(axis='x', linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.savefig('grafico_coleccion.png', dpi=300) # Guarda el gráfico para la entrega
    plt.show()

In [ ]:
if 'df' in locals() and not df.empty:
    cartas_con_imagen = df[df['imagen_url'] != ''].head(8)

    if not cartas_con_imagen.empty:
        cols = 4
        rows = (len(cartas_con_imagen) + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(12, 4 * rows))

        if rows == 1 and cols == 1:
            axes = [axes]
        else:
            axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

        for i, (_, row) in enumerate(cartas_con_imagen.iterrows()):
            try:
                res = requests.get(row['imagen_url'], timeout=5)
                img = Image.open(BytesIO(res.content))
                axes[i].imshow(img)
                axes[i].set_title(f"{row['nombre']}\nx{row['cantidad']} copias", fontsize=10)
                axes[i].axis('off')
            except Exception as e:
                axes[i].text(0.5, 0.5, "Error Imagen", ha='center')
                axes[i].axis('off')

        # Ocultar cuadros vacíos si hay menos de 8 cartas
        for j in range(i + 1, len(axes)):
            axes[j].axis('off')

        plt.tight_layout()
        plt.show()